# Stadt Zürich — Meteo Master Dataset

Erstellung des konsolidierten Wetter-Datensatzes aus drei Quellen für die spätere Korrelation mit den Tram-IST-Daten.

| Quelle | Beitrag | Granularität |
| :--- | :--- | :--- |
| **UGZ Stampfenbachstrasse** | Temperatur, Regen-Dauer, Wind, Strahlung, Luftdruck | Stündlich (h1) |
| **Wapo Mythenquai** | Niederschlagsmenge (mm) | 10-Min → stündlich aggregiert |
| **ERZ Überschwemmungsmeldungen** | Ereignis-Intensität | Täglich → stündlich expandiert |

**Output:** `data/interim/vbz/meteo/meteo-final-export.parquet`

# Konsolidierung Strategisches Meteo-Dataset für Tram-IST-Analyse

Dieses Dokument beschreibt die Konsolidierung verschiedener Wetterdatenquellen der Stadt Zürich. Ziel ist ein hochpräziser "Master-Wetter-Datensatz", um Korrelationen zwischen Witterung und Tram-Verspätungen zu analysieren.

---

## Inhaltsverzeichnis

1. [Ausgangslage & Zielsetzung](#1-ausgangslage--zielsetzung)
2. [Stationen & Standort-Entscheidung](#2-stationen--standort-entscheidung)
3. [Transformations- & Merge-Strategie](#3-transformations---merge-strategie)  
  
  
---

## 1. Ausgangslage & Zielsetzung

Die Standard-Meteodaten der Stadt Zürich liefern präzise Stundenwerte zur **Dauer** von Niederschlagsereignissen, lassen jedoch Informationen zur **Intensität** (Niederschlagsmenge in mm) und zu Winterereignissen (Schnee) vermissen. Durch die Kombination von UGZ-Daten, Wasserschutzpolizei-Messungen und Einsatzmeldungen schließen wir diese Lücken.

### Quellen

Verweis zur vbz-data übersicht...

### Lokale Dateipfade

Verweis zur vbz-data übersicht...


---

### Datenwörterbucher

In [1]:
import sys, os 
sys.path.append(os.path.abspath('../../../src'))
from doc_loader import show_doc

In [2]:
show_doc('meteo_ugz')

### Datenwörterbuch: UGZ Meteodaten (Referenz-Station Stampfenbachstrasse)

Stündliche Mittelwerte (**Intervall h1**) — Basis für den späteren Join mit den Tram-IST-Daten.

| Parameter | Name | Einheit | Relevanz für das Tramprojekt |
| :--- | :--- | :--- | :--- |
| **T** | Temperatur | °C | Heiz-/Kühlbedarf, Fahrgastaufkommen, Technikbelastung. Basis für `is_snow`. |
| **RainDur** | Niederschlagsdauer | min | **Kritisch:** Einfluss auf Bremswege, Verspätungen, Umstieg auf ÖV. |
| **StrGlo** | Globalstrahlung | W/m² | Hitzeentwicklung in Fahrzeugen, "Schönwetter-Effekt". |
| **Hr** | Rel. Luftfeuchtigkeit | %Hr | Komfortindex (Schwüle). |
| **p** | Luftdruck | hPa | Meteorologische Basisdaten. |
| **WVs / WVv** | Windgeschwindigkeit / -vektor | m/s | Sturmereignisse (mögliche Streckenblockaden). |

> **Lücke:** Der UGZ-Datensatz enthält `RainDur` (Niederschlagsdauer), aber keine Niederschlagsmenge (mm). Diese Lücke wird durch den Merge mit der Wapo-Station Mythenquai geschlossen.

In [3]:
show_doc('meteo_wapo')

### Datenwörterbuch: Wapo Wetterstationen (Mythenquai — Primär / Tiefenbrunnen — Backup)

Hochauflösende Messungen (~10-Minuten-Intervall) direkt vom Seeufer. Liefert die essenzielle **Niederschlagsmenge** als Ergänzung zu den UGZ-Stundenwerten.

| Parameter | Name | Einheit | Beschreibung |
| :--- | :--- | :--- | :--- |
| **timestamp_cet** | Zeitstempel | ISO8601 | Datum und Uhrzeit der Messung (CET). |
| **precipitation** | **Niederschlagsmenge** | **mm** | **Summe des gefallenen Niederschlags (Regen/Schnee-Äquivalent). Kernergänzung zum UGZ-Datensatz.** |

In [4]:
show_doc('meteo_erz')


### Datenwörterbuch: Überschwemmungsmeldungen (ERZ / Schutz & Rettung)

Indikator für extreme Wetterauswirkungen auf die städtische Infrastruktur.

| Spalte | Name | Beschreibung |
| :--- | :--- | :--- |
| **stznr** | Statistische Zone | Nummer der statistischen Zone in der Stadt Zürich. |
| **datum_ereignis** | Ereignisdatum | Kalendertag des Vorfalls (Format: YYYY-MM-DD). |
| **anz_ereignisse** | Anzahl Meldungen | Anzahl der eingegangenen Notrufe/Meldungen pro Tag und Zone. |


## 2. Stationen & Standort-Entscheidung

### UGZ Messstationen

| Station | Lage | Besonderheiten | Entscheidung |
| :--- | :--- | :--- | :--- |
| **Zch_Stampfenbachstrasse** | Stadtzentrum, 445 m.ü.M. | Vollständigster Datensatz inkl. **Globalstrahlung**. Zentral nahe HB. | **Referenz-Station** |
| **Zch_Schimmelstrasse** | Stadtzentrum, 413 m.ü.M. | Keine Globalstrahlung. Stark befahren. | Nur Backup |
| **Zch_Rosengartenstrasse** | Hauptverkehrsachse, 433 m.ü.M. | Starke Verkehrsexposition. | Nur Backup |
| **Zch_Heubeeribüel** | Höhere Lage, 610 m.ü.M. | Nur Luftdruck, für Tram-Ebene ungeeignet. | Aussortiert |

**Begründung:** Die **Stampfenbachstrasse** liegt zentral nahe des Hauptbahnhofs (Knotenpunkt des Tramnetzes) und liefert alle relevanten Parameter inklusive Globalstrahlung — ideale Referenz für das städtische Klima.

### Wapo Wetterstationen

| Station | Lage | Entscheidung |
| :--- | :--- | :--- |
| **Mythenquai** | Seeufer, zentral nahe HB/Bellevue | **Primär** — näher am Stadtzentrum, liefert `precipitation` (mm). |
| **Tiefenbrunnen** | Seeufer, östlich | Nur Backup — weiter vom Tramnetz-Schwerpunkt entfernt. |

---

## 3. Transformation

### Pivoting: Long → Wide

Die UGZ-Rohdaten kommen im **Long-Format** (jeder Messwert eine eigene Zeile):

- **Ist-Zustand:** 1 Stunde × 6 Parameter = 6 Zeilen
- **Ziel-Zustand:** 1 Stunde = 1 Zeile (Parameter als Spalten nebeneinander)

Erst im Wide-Format lassen sich Korrelationen zwischen Parametern (z.B. Temperatur vs. Verspätung) berechnen und die Daten effizient mit den Tram-IST-Daten mergen.

### Quellenlogik: Was kommt woher?

| Messgrösse | Quelle | Begründung |
| :--- | :--- | :--- |
| Temperatur (`T`) | **UGZ Stampfenbachstrasse** | Referenz-Station, repräsentativ für das gesamte Stadtnetz. |
| Niederschlagsdauer (`RainDur`) | **UGZ Stampfenbachstrasse** | Einzige Quelle für Ereignisdauer in Minuten. |
| Niederschlagsmenge (`precipitation_mm`) | **Wapo Mythenquai** | Einzige Quelle mit mm-Werten — schließt Lücke des UGZ-Datensatzes. |
| Globalstrahlung, Wind, Luftdruck | **UGZ Stampfenbachstrasse** | Vollständigster UGZ-Datensatz. |
| Extremereignis-Indikator (`flood_intensity`) | **ERZ Überschwemmungsmeldungen** | Validierung von Infrastrukturauswirkungen durch Starkregen. |

> Alle Temperaturen und abgeleiteten Features basieren ausschließlich auf dem UGZ-Wert `T` der **Stampfenbachstrasse**. Die Wapo-Temperatur fließt nicht als Referenz ein — Mythenquai wird nur für `precipitation_mm` genutzt.

### Pipeline: Prozess-Schritte

#### Schritt 1 — Vorbereitung UGZ (Pivot)
- Zusammenführung der drei Jahres-CSVs (2023–2025)
- Filterung auf Referenz-Station Stampfenbachstrasse
- Reduktion auf Spalten: `Datum`, `Parameter`, `Wert`
- Pivot Long → Wide (1 Zeile pro Stunde)
- Zeitstempel normalisiert zu `date_time` (`datetime64[us]`, CET, volle Stunde)
- Spalten auf englische Bezeichnungen umbenannt

#### Schritt 2 — Vorbereitung Wapo (Resampling)
- Filterung auf Periode 2023–2025
- Aggregation 10-Min-Intervalle → Stundenwerte: `precipitation` als **Summe**
- Zeitstempel normalisiert zu `date_time` (tz-naiv)
- Umbenennung zu `precipitation_mm`

#### Schritt 3 — Vorbereitung ERZ (Tages-Expansion)
- Aggregation über alle Zonen: `anz_ereignisse` summiert pro Tag
- Umbenennung zu `flood_intensity`
- Join später über Datumsteil (jede Stunde eines Ereignistages erhält den Tageswert)

#### Schritt 4 — Zusammenführung (Left Join)
- UGZ als Gerüst (linke Tabelle)
- Wapo-Join über `date_time` (stündlich)
- ERZ-Join über Datumsteil (`date_time.dt.normalize()`)
- `flood_intensity`: NaN → 0

> **Feature Engineering** (`is_snow`, `flood_alert`, Regen-Kategorien) erfolgt in der EDA, nachdem die Datenverteilung analysiert wurde.

---

In [9]:
import pandas as pd
from pathlib import Path

# ROOT automatisch finden — egal wo das Notebook liegt
for _p in [Path.cwd()] + list(Path.cwd().parents):
    if (_p / 'data' / 'raw').exists():
        ROOT = _p
        break

METEO_PATH = str(ROOT / 'data' / 'raw' / 'vbz' / 'meteo') + '/'
print(f'METEO_PATH: {METEO_PATH}')


METEO_PATH: /Users/kaywiegand/Workspace/sf_data-research/data/raw/vbz/meteo/


In [10]:
print(f'METEO_PATH = {METEO_PATH}')

METEO_PATH = /Users/kaywiegand/Workspace/sf_data-research/data/raw/vbz/meteo/


---

## Schritt 1 — Vorbereitung UGZ Stampfenbachstrasse

Drei Jahres-CSVs laden, auf die Referenz-Station filtern, pivoten und Zeitstempel normalisieren.

In [11]:
# Laden der drei Jahres-CSVs und Zusammenführung
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
df_ugz_23 = pd.read_csv(METEO_PATH + 'ugz/ugz_ogd_meteo_h1_2023.csv')
df_ugz_24 = pd.read_csv(METEO_PATH + 'ugz/ugz_ogd_meteo_h1_2024.csv')
df_ugz_25 = pd.read_csv(METEO_PATH + 'ugz/ugz_ogd_meteo_h1_2025.csv')

df_ugz = pd.concat([df_ugz_23, df_ugz_24, df_ugz_25], ignore_index=True)

# Filterung auf Referenz-Station
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
df_ugz = df_ugz[df_ugz['Standort'] == 'Zch_Stampfenbachstrasse']

# Reduktion auf die drei relevanten Spalten (Datum, Parameter, Wert)
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
df_ugz = df_ugz[['Datum', 'Parameter', 'Wert']]

# Pivot: Long → Wide (1 Zeile pro Stunde, Parameter als Spalten)
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
df_ugz = df_ugz.pivot(index='Datum', columns='Parameter', values='Wert').reset_index()

# Zeitstempel normalisieren: UTC-aware → CET → tz-naiv → volle Stunde
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
df_ugz['Datum'] = (
    pd.to_datetime(df_ugz['Datum'], utc=True)
    .dt.tz_convert('Europe/Zurich')
    .dt.tz_localize(None)
    .dt.floor('h')
)

# Spalten auf englische Bezeichnungen umbenennen
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
df_ugz = df_ugz.rename(columns={
    'Datum': 'date_time',
    'T':     'temperature',
    'Hr':    'humidity',
    'p':     'air_pressure',
    'RainDur': 'rain_duration',
    'StrGlo':  'global_radiation',
    'WD':    'wind_direction',
    'WVs':   'wind_speed',
    'WVv':   'wind_speed_vector',
})

df_ugz.info()

<class 'pandas.DataFrame'>
RangeIndex: 26304 entries, 0 to 26303
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   date_time          26304 non-null  datetime64[us]
 1   humidity           26261 non-null  float64       
 2   rain_duration      26277 non-null  float64       
 3   global_radiation   26272 non-null  float64       
 4   temperature        26261 non-null  float64       
 5   wind_direction     26283 non-null  float64       
 6   wind_speed         26272 non-null  float64       
 7   wind_speed_vector  26283 non-null  float64       
 8   air_pressure       26278 non-null  float64       
dtypes: datetime64[us](1), float64(8)
memory usage: 1.8 MB


---

## Schritt 2 — Vorbereitung Wapo Mythenquai

Niederschlagsmenge (mm) aus den 10-Minuten-Messungen auf Stundenbasis aggregieren.

| Spalte | Aggregation | Begründung |
| :--- | :--- | :--- |
| `precipitation` | **Summe** | Gesamter Niederschlag der Stunde — korrekte Abbildung von Stark- vs. Dauerregen. |

In [12]:
# Laden der Wapo-Daten (Mythenquai)
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
df_wapo = pd.read_parquet(METEO_PATH + 'wapo/messwerte_mythenquai_seit2007-heute.parquet')

# Reduktion auf relevante Spalten
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
df_wapo = df_wapo[['timestamp_cet', 'precipitation']]

# Filterung auf Periode 2023–2025
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
date_mask = (df_wapo['timestamp_cet'] >= '2023-01-01') & (df_wapo['timestamp_cet'] <= '2025-12-31')
df_wapo = df_wapo[date_mask]

# Zeitstempel: tz-naiv
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
df_wapo['timestamp_cet'] = pd.to_datetime(df_wapo['timestamp_cet']).dt.tz_localize(None)

# Sicherstellung numerisches Format (parquet kann object-Typ liefern)
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
df_wapo['precipitation'] = pd.to_numeric(df_wapo['precipitation'], errors='coerce')

# Resampling: 10-Min → Stundenbasis (Niederschlag summieren)
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
df_wapo = (
    df_wapo
    .set_index('timestamp_cet')
    .resample('h')['precipitation']
    .sum()
    .reset_index()
    .rename(columns={
        'timestamp_cet': 'date_time',
        'precipitation': 'precipitation_mm'
    })
)

df_wapo.head()

,date_time,precipitation_mm
0,2023-01-01 00:00:00,0.0
1,2023-01-01 01:00:00,0.0
2,2023-01-01 02:00:00,0.0
3,2023-01-01 03:00:00,0.0
4,2023-01-01 04:00:00,0.0


---

## Schritt 3 — Vorbereitung ERZ Überschwemmungsmeldungen

Da nur Tageswerte vorliegen, wird `anz_ereignisse` als `flood_intensity` für jede Stunde eines betroffenen Tages zugewiesen.

> Eine Binarisierung zu `flood_alert` (0/1) erfolgt in der EDA, sobald ein sinnvoller Schwellenwert aus der Datenverteilung abgeleitet werden kann.

In [13]:
# Laden der Überschwemmungsmeldungen
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
df_erz = pd.read_csv(METEO_PATH + 'erz/erz_ent_ueberschwemmungsmeldungen.csv')

# Aggregation auf Tagesebene: Summe aller Meldungen über alle Zonen
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
df_erz['datum_ereignis'] = pd.to_datetime(df_erz['datum_ereignis'])

df_erz_daily = (
    df_erz
    .groupby('datum_ereignis')['anz_ereignisse']
    .sum()
    .reset_index()
    .rename(columns={
        'datum_ereignis': 'date_time',
        'anz_ereignisse': 'flood_intensity'
    })
)

df_erz_daily.head()

,date_time,flood_intensity
0,1976-06-10,1
1,1976-06-19,1
2,1977-06-18,1
3,1977-06-30,1
4,1977-07-08,16


---

## Schritt 4 — Zusammenführung (Left Join)

UGZ als Gerüst — Wapo und ERZ werden rechtsseitig angehängt.

```
df_ugz  LEFT JOIN  df_wapo     ON date_time          (stündlich)
        LEFT JOIN  df_erz_daily ON date_time (Datum)  (täglich → stündlich)
```

In [14]:
# Join 1: UGZ + Wapo (stündlicher Schlüssel)
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
df_master = pd.merge(df_ugz, df_wapo, on='date_time', how='left')

# Join 2: + ERZ (Schlüssel = Datumsteil, ohne Uhrzeit)
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
df_master['_datum'] = df_master['date_time'].dt.normalize()

df_master = (
    pd.merge(df_master, df_erz_daily,
             left_on='_datum', right_on='date_time', how='left')
    .drop(columns=['_datum', 'date_time_y'])
    .rename(columns={'date_time_x': 'date_time'})
)

# Kein Ereignis an diesem Tag → 0 (kein NaN im finalen Dataset)
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
df_master['flood_intensity'] = df_master['flood_intensity'].fillna(0).astype(int)

display(df_master)

,date_time,humidity,rain_duration,global_radiation,temperature,wind_direction,wind_speed,wind_speed_vector,air_pressure,precipitation_mm,flood_intensity
0,2023-01-01 00:00:00,72.29,0.0,0.01,11.57,169.49,1.95,1.69,971.62,0.0,0
1,2023-01-01 01:00:00,63.66,0.0,0.02,13.47,205.89,3.40,2.77,971.86,0.0,0
2,2023-01-01 02:00:00,68.85,0.0,0.02,12.39,149.11,1.98,1.49,971.76,0.0,0
3,2023-01-01 03:00:00,70.72,0.0,0.02,11.69,157.08,1.79,1.46,972.01,0.0,0
4,2023-01-01 04:00:00,70.45,0.0,0.02,11.55,178.54,2.98,2.74,972.10,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...
26299,2025-12-31 19:00:00,53.09,0.0,0.02,-1.41,34.52,0.65,0.26,971.74,NaN,0
26300,2025-12-31 20:00:00,58.78,0.0,0.01,-1.80,297.09,0.39,0.21,971.53,NaN,0
26301,2025-12-31 21:00:00,62.60,0.0,0.02,-2.14,280.12,0.43,0.37,971.27,NaN,0
26302,2025-12-31 22:00:00,64.58,0.0,0.01,-2.63,21.68,1.16,1.11,971.13,NaN,0


In [15]:

show_doc('meteo')


## Datenwörterbuch — Meteo-Master-Dataset

| Spalte | Dtype | Einheit | Quelle | Beschreibung |
| :--- | :--- | :--- | :--- | :--- |
| **date_time** | datetime64[us] | — | UGZ | Zeitstempel, stündlich, CET, tz-naiv |
| **temperature** | float64 | °C | UGZ Stampfenbachstrasse | Lufttemperatur. Basis für `is_snow` in der EDA. |
| **humidity** | float64 | %Hr | UGZ Stampfenbachstrasse | Relative Luftfeuchtigkeit. |
| **air_pressure** | float64 | hPa | UGZ Stampfenbachstrasse | Luftdruck. |
| **rain_duration** | float64 | min | UGZ Stampfenbachstrasse | Niederschlagsdauer pro Stunde. |
| **global_radiation** | float64 | W/m² | UGZ Stampfenbachstrasse | Globalstrahlung (Sonneneinstrahlung). |
| **wind_direction** | float64 | ° | UGZ Stampfenbachstrasse | Windrichtung (0–360°). |
| **wind_speed** | float64 | m/s | UGZ Stampfenbachstrasse | Windgeschwindigkeit (skalar). |
| **wind_speed_vector** | float64 | m/s | UGZ Stampfenbachstrasse | Windgeschwindigkeit (Vektor). |
| **precipitation_mm** | float64 | mm | Wapo Mythenquai | Niederschlagsmenge, stündlich summiert. NaN wenn keine Wapo-Messung. |
| **flood_intensity** | int64 | Anzahl | ERZ | Summe Überschwemmungsmeldungen des Tages über alle Zonen. 0 = kein Ereignis. |

> **EDA-Aufgaben:**   
* `is_snow` (temperature < 1°C & rain_duration > 0),   
* `flood_alert` (Schwellenwert aus Verteilung),   
* Regen-Kategorien aus `precipitation_mm`.  

---

## Export

Ausgabe als Parquet — erhält Datentypen, schnell einlesbar, bis zu 90% kleiner als CSV.

In [19]:
from pathlib import Path

# Ausgabe-Verzeichnis anlegen (falls nicht vorhanden)
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Path('../../../data/interim/vbz/meteo/').mkdir(parents=True, exist_ok=True)

# Export
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
df_master.to_parquet('../../../data/interim/vbz/meteo/meteo-master.parquet', index=False)

print(f'Exportiert: {len(df_master):,} Zeilen, {len(df_master.columns)} Spalten')
print(f'Zeitraum:   {df_master["date_time"].min()} bis {df_master["date_time"].max()}')

Exportiert: 26,304 Zeilen, 11 Spalten
Zeitraum:   2023-01-01 00:00:00 bis 2025-12-31 23:00:00


In [4]:
import pandas as pd
print(pd.read_parquet('../../../data/interim/vbz/meteo/meteo-master.parquet').shape)

(26304, 11)
